# BA-CNO on CylinderFlow: Colab runner

Runs the study plan on a Colab/Kaggle GPU (`docs/STUDY_PLAN.md` section 6):

1. **Setup:** clone, install, download + convert CylinderFlow to the VM's local disk.
2. **Day 2, timing:** a short run per config → measured it/s → one step budget for every config.
3. **Days 3–5, training:** one config per session. Checkpoints go to Google Drive, so a disconnected session resumes from `last.pt` when you rerun the cell.
4. **Evaluation:** full 599-step rollouts on the test split.

Data stays on the VM disk (re-downloaded each session, ~3 min for train from GCS). Only checkpoints and results go to Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import psutil, shutil
print(f"RAM {psutil.virtual_memory().total / 2**30:.1f} GiB, disk free {shutil.disk_usage('/content').free / 2**30:.0f} GiB")

## 1. Setup

In [ ]:
REPO = "https://github.com/ramtarun02/BA-CNO.git"   # private repo: use https://<token>@github.com/...
BRANCH = "ba-cno-benchmarks"
DATA = "/content/data/cylinder_flow"

from google.colab import drive
drive.mount("/content/drive")
RUNS = "/content/drive/MyDrive/ba_cno/outputs/cylinder"   # checkpoints survive disconnects

!git clone -q -b {BRANCH} {REPO} /content/BA-CNO || git -C /content/BA-CNO pull -q
%cd /content/BA-CNO
!pip install -q -e .

In [ ]:
# Train is 13.6 GB. Converted trajectories are memory-mapped during training, so RAM use stays small.
# The raw TFRecord is deleted after conversion to keep disk use at ~1x the dataset.
import os
for split in ["valid", "test", "train"]:
    if not os.path.isdir(f"{DATA}/{split}"):
        !bash scripts/cylinder/download.sh {DATA}/raw {split}
        !python scripts/cylinder/convert.py --raw {DATA}/raw --out {DATA} --splits {split}
        !rm -f {DATA}/raw/{split}.tfrecord
!ls {DATA}/train | wc -l

## 2. Day 2: timing and step budget

Trains every config for a short run at the real batch size and records the steady-state it/s (second half of the run) in `timing.json`. `budget.py` then picks the largest round step count that fits `HOURS_PER_CONFIG` for the **slowest** config, so all five get the same budget.

In [ ]:
CONFIGS = ["mgn", "ba_cno_soft", "ba_cno_hard", "ba_cno_none_fluxdec", "ba_cno_hard_fluxdec"]
TIMING_STEPS = 1000
HOURS_PER_CONFIG = 20

for c in CONFIGS:
    !python scripts/cylinder/train.py --config configs/cylinder/{c}.yaml --override \
        output_dir={RUNS}/timing/{c} data.root={DATA} train.max_steps={TIMING_STEPS} train.log_every=250 train.ckpt_every=100000
!python scripts/cylinder/budget.py --runs {RUNS}/timing/* --hours-per-config {HOURS_PER_CONFIG} --out {RUNS}/timing/budget.json

## 3. Days 3–5: training (one config per session)

Set `CONFIG`, run the cell, and rerun it after any disconnect: it resumes from `{RUNS}/{CONFIG}/last.pt`. To run configs in parallel, open this notebook in several sessions or accounts with a different `CONFIG` in each.

Keep `MAX_STEPS` the same for every config: it is the locked budget from section 2, so take it from `budget.json`. Changing it mid-run also changes the LR schedule.

In [ ]:
import json
CONFIG = "mgn"
MAX_STEPS = json.load(open(f"{RUNS}/timing/budget.json"))["budget_steps"]
print(CONFIG, MAX_STEPS)
!python scripts/cylinder/train.py --config configs/cylinder/{CONFIG}.yaml --override \
    output_dir={RUNS}/{CONFIG} data.root={DATA} train.max_steps={MAX_STEPS} train.log_every=1000

## 4. Evaluation (Day 6): 599-step rollouts on the 100 test trajectories

In [ ]:
!python scripts/cylinder/evaluate.py --run {RUNS}/{CONFIG} --data-root {DATA} --device cuda